# 02 — Geocode container addresses

This notebook takes the cleaned LIMASAM addresses and geocodes each one using Nominatim (OpenStreetMap), producing point coordinates ready for spatial analysis.

**Input:** `data/interim/containers_cleaned.csv` (480 cleaned addresses)
**Output:** `data/processed/containers_geocoded.csv` (with `lat`, `lon`, `match_type`, `geocoding_quality`)

## Strategy

1. Use Nominatim, respecting its rate limit (1 request per second) and policy (custom user-agent).
2. For each address, capture the returned `match_type` (`house` / `road` / `place` etc.) to assess geocoding quality.
3. Tag each result as:
   - **`high`** — `house_number` request returned a precise `house` or `building` match
   - **`medium`** — returned a street or intersection match
   - **`low`** — only a generic place or POI was found
   - **`failed`** — no result
4. Validate that all points fall inside Málaga's bounding box; flag any outliers.

## Notes

- ~480 requests × 1 second = ~8–10 minutes runtime.
- Results are cached: if you re-run the notebook, addresses already geocoded are skipped.

In [1]:
import time
from pathlib import Path

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)  # create if missing

# Files
INPUT_PATH = DATA_INTERIM / "containers_cleaned.csv"
OUTPUT_PATH = DATA_PROCESSED / "containers_geocoded.csv"

print(f"Input file:  {INPUT_PATH}")
print(f"Output file: {OUTPUT_PATH}")
print(f"Input exists: {INPUT_PATH.exists()}")

Input file:  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\containers_cleaned.csv
Output file: c:\Users\user\Documents\Projects\malaga-textile-access\data\processed\containers_geocoded.csv
Input exists: True


In [2]:
df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(df)} addresses")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Loaded 480 addresses
Columns: ['address_raw', 'address_type', 'address_clean']

First 3 rows:


,address_raw,address_type,address_clean
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil..."


In [3]:
# Set up Nominatim with a custom user-agent (their policy requires identifying the app)
geolocator = Nominatim(
    user_agent="malaga-textile-access-project (lidavaynberg@gmail.com)",
    timeout=10,
)

# Wrap in a rate limiter: at most 1 request per second
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,   # 1.1s to be safe
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False,
)

# Pick one address from each type to test
test_samples = (
    df.groupby("address_type")
      .first()
      .reset_index()
)
print("Testing geocoder on one address of each type:\n")

for _, row in test_samples.iterrows():
    addr = row["address_clean"]
    addr_type = row["address_type"]
    print(f"[{addr_type}]  {addr}")
    
    location = geocode(addr)
    if location:
        print(f"  → ({location.latitude:.5f}, {location.longitude:.5f})")
        print(f"  → matched: {location.address}")
        print(f"  → raw type: {location.raw.get('type')}, class: {location.raw.get('class')}")
    else:
        print(f"  → NOT FOUND")
    print()

Testing geocoder on one address of each type:

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  → NOT FOUND

[intersection]  Andarax and Camino de San Rafael, Málaga, Spain
  → NOT FOUND

[street_only]  Calle Almogía, Málaga, Spain
  → (36.71439, -4.45309)
  → matched: Calle Almogía, Polígono Carretera de Cártama, Los Corazones, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29010, España
  → raw type: secondary, class: highway



In [4]:
# Test variations for a house_number address
test_address_variants = [
    "Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain",
    "Avenida de la Paloma, 36, Carretera de Cádiz, 29003, Málaga, Spain",
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Avenida de la Paloma 36, Málaga, Spain",
    "Av de la Paloma 36, Málaga",
]

print("Testing variations of the same address:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Testing variations of the same address:

→ Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Carretera de Cádiz, 29003, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Avenida de la Paloma, 36, Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Avenida de la Paloma 36, Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Av de la Paloma 36, Málaga
  ✓ (36.49128, -4.70622)  type=residential



In [5]:
# Málaga bounding box (approximate, generous)
# (west, south, east, north) — covering the city and immediate surroundings
malaga_bbox = [(-4.55, 36.65), (-4.30, 36.78)]
#                ↑ SW corner       ↑ NE corner

def geocode_in_malaga(address):
    """Geocode an address, restricting results to Málaga's bounding box and Spain."""
    return geocode(
        address,
        country_codes="es",          # only Spain
        viewbox=malaga_bbox,
        bounded=True,                 # strict: ONLY return matches inside the box
    )

# Re-test the same variations, now restricted to Málaga
test_address_variants = [
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Avenida de la Paloma 36, Málaga, Spain",
    "Av de la Paloma 36, Málaga",
]

print("Re-testing with Málaga bounding box restriction:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode_in_malaga(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}")
        print(f"    matched: {loc.address[:100]}...")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Re-testing with Málaga bounding box restriction:

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma 36, Málaga, Spain
  ✗ NOT FOUND

→ Av de la Paloma 36, Málaga
  ✗ NOT FOUND



In [6]:
# Search just the street, without house number
test_streets = [
    "Avenida de la Paloma, Málaga",
    "Calle Goya, Málaga",
    "Calle Conde de Guadalhorce, Málaga",
]

print("Testing street-only searches with bounding box:\n")
for street in test_streets:
    print(f"→ {street}")
    loc = geocode_in_malaga(street)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})")
        print(f"    matched: {loc.address[:120]}")
    else:
        print(f"  ✗ NOT FOUND inside Málaga bbox")
    print()

Testing street-only searches with bounding box:

→ Avenida de la Paloma, Málaga
  ✗ NOT FOUND inside Málaga bbox

→ Calle Goya, Málaga
  ✗ NOT FOUND inside Málaga bbox

→ Calle Conde de Guadalhorce, Málaga
  ✗ NOT FOUND inside Málaga bbox



In [7]:
# Test 1: search WITHOUT any restriction — does Nominatim find Calle Goya at all?
print("=== Test 1: no restrictions ===")
loc = geocode("Calle Goya, Málaga, Spain")
if loc:
    print(f"  Found at: ({loc.latitude:.5f}, {loc.longitude:.5f})")
    print(f"  Full address: {loc.address}")
else:
    print("  NOT FOUND")

# Test 2: same but only country restriction (no bbox)
print("\n=== Test 2: only country=es ===")
loc = geocode("Calle Goya, Málaga, Spain", country_codes="es")
if loc:
    print(f"  Found at: ({loc.latitude:.5f}, {loc.longitude:.5f})")
    print(f"  Full address: {loc.address}")
else:
    print("  NOT FOUND")

# Show what bbox I'm passing
print("\n=== Current bbox parameter ===")
print(f"malaga_bbox = {malaga_bbox}")
print(f"This is interpreted as: SW corner = {malaga_bbox[0]}, NE corner = {malaga_bbox[1]}")
print(f"  i.e. longitude {malaga_bbox[0][0]} to {malaga_bbox[1][0]}")
print(f"  and  latitude  {malaga_bbox[0][1]} to {malaga_bbox[1][1]}")

=== Test 1: no restrictions ===
  Found at: (36.70537, -4.43710)
  Full address: Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

=== Test 2: only country=es ===
  Found at: (36.70579, -4.43760)
  Full address: Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

=== Current bbox parameter ===
malaga_bbox = [(-4.55, 36.65), (-4.3, 36.78)]
This is interpreted as: SW corner = (-4.55, 36.65), NE corner = (-4.3, 36.78)
  i.e. longitude -4.55 to -4.3
  and  latitude  36.65 to 36.78


In [8]:
# CORRECT format: viewbox expects (latitude, longitude) tuples
# Málaga bounding box (generous, covers city + immediate surroundings)
malaga_bbox = [(36.65, -4.55), (36.78, -4.30)]
#                ↑ SW (lat, lon)   ↑ NE (lat, lon)

def geocode_in_malaga(address):
    """Geocode an address, restricting results to Málaga's bounding box and Spain."""
    return geocode(
        address,
        country_codes="es",
        viewbox=malaga_bbox,
        bounded=True,
    )

# Re-test
test_address_variants = [
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Calle Goya, 7, Málaga, Spain",
    "Calle Goya, Málaga, Spain",
    "Calle Conde de Guadalhorce, 6, Málaga, Spain",
    "Andarax and Camino de San Rafael, Málaga, Spain",
    "Calle Almogía, Málaga, Spain",
]

print("Re-testing with FIXED bbox order:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode_in_malaga(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}, class={loc.raw.get('class')}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Re-testing with FIXED bbox order:

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Málaga, Spain
  ✗ NOT FOUND

→ Calle Goya, 7, Málaga, Spain
  ✓ (36.70553, -4.43741)  type=house, class=place
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

→ Calle Goya, Málaga, Spain
  ✓ (36.70537, -4.43710)  type=tertiary, class=highway
    Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

→ Calle Conde de Guadalhorce, 6, Málaga, Spain
  ✓ (36.71332, -4.44156)  type=house, class=place
    6A, Calle Conde de Guadalhorce, Cruz del Humilladero, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, Málaga, Andaluc

→ Andarax and Camino de San Rafael, Málaga, Spain
  ✗ NOT FOUND

→ Calle Almogía, Málaga, Spain
  ✓ (36.71424, -4.45314)  type=secondary, class=highway
    Calle Almogía, La Barriguilla, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, M

In [9]:
import re

def simplify_address(address: str) -> str:
    """Remove 'de la', 'del', 'de' between street type and street name."""
    # Match patterns like 'Avenida de la X', 'Calle del Y', 'Calle de Z'
    # Be careful: only remove the article right after the street type
    return re.sub(
        r"\b(Avenida|Calle|Plaza|Camino|Carretera|Pasaje|Paseo|Glorieta)\s+(de\s+la|de\s+las|del|de\s+los|de)\s+",
        r"\1 ",
        address,
        flags=re.IGNORECASE,
    )


def strip_house_number(address: str) -> str:
    """Remove the first numeric block (house number) from an address."""
    # Pattern: ', 36-38, ...' or ', 36, ...' or ' 36, ...'
    return re.sub(r",?\s*\d+[\d\-A-Za-z]*\s*,", ",", address, count=1).strip(", ")


def geocode_with_fallbacks(address_clean: str, address_type: str):
    """Try several formats; return (location, quality_tag)."""
    
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        match_class = loc.raw.get("class")
        if match_class == "place" and loc.raw.get("type") == "house":
            return loc, "high", "original"
        else:
            return loc, "medium", "original"
    
    # Attempt 2: simplified (remove "de la", "del" etc.)
    simplified = simplify_address(address_clean)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            match_class = loc.raw.get("class")
            if match_class == "place" and loc.raw.get("type") == "house":
                return loc, "high", "simplified"
            else:
                return loc, "medium", "simplified"
    
    # Attempt 3: strip house number, try as street
    no_number = strip_house_number(address_clean)
    if no_number != address_clean:
        loc = geocode_in_malaga(no_number)
        if loc:
            return loc, "low", "street_only_fallback"
    
    # Attempt 4: strip house number AND simplify
    no_num_simplified = simplify_address(strip_house_number(address_clean))
    if no_num_simplified not in (address_clean, simplified, no_number):
        loc = geocode_in_malaga(no_num_simplified)
        if loc:
            return loc, "low", "street_only_simplified_fallback"
    
    return None, "failed", "none"


# Test on the problematic samples
test_cases = [
    ("Avenida de la Paloma, 36, 29003, Málaga, Spain", "house_number"),
    ("Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain", "house_number"),
    ("Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Andarax and Camino de San Rafael, Málaga, Spain", "intersection"),
    ("Calle Almogía, Málaga, Spain", "street_only"),
]

print("Testing fallback geocoding strategy:\n")
for addr, addr_type in test_cases:
    print(f"[{addr_type}]  {addr}")
    loc, quality, method = geocode_with_fallbacks(addr, addr_type)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  quality={quality}, method={method}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ FAILED")
    print()

Testing fallback geocoding strategy:

[house_number]  Avenida de la Paloma, 36, 29003, Málaga, Spain
  ✓ (36.70166, -4.44376)  quality=high, method=simplified
    36, Avenida Paloma, San Carlos Condote, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, Espa

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70066, -4.44184)  quality=medium, method=simplified
    Avenida Paloma, Girón, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, España

[house_number]  Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain
  ✓ (36.70553, -4.43741)  quality=high, method=original
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

[house_number]  Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70023, -4.43988)  quality=medium, method=original
    Calle Federico García Lorca, 25 Años de Paz, Carretera de Cádiz,

In [10]:
def split_intersection(address: str):
    """If the address contains ' and ', return the first and second street names."""
    # Look for ' and ' as our standardized intersection separator
    parts = re.split(r"\s+and\s+", address, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) == 2:
        # First part is street A; second part is "street B, Málaga, Spain" — strip the tail
        street_a = parts[0].strip()
        # Reconstruct street B with the Málaga suffix
        street_b_raw = parts[1].strip()
        # Take only the street name from street B (before the first comma)
        street_b_name = street_b_raw.split(",")[0].strip()
        return street_a + ", Málaga, Spain", street_b_name + ", Málaga, Spain"
    return None, None


def geocode_with_fallbacks(address_clean: str, address_type: str):
    """Try several formats; return (location, quality_tag, method_used)."""
    
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        match_class = loc.raw.get("class")
        match_type = loc.raw.get("type")
        if match_class == "place" and match_type == "house":
            return loc, "high", "original"
        elif address_type == "street_only":
            # Street-only addresses cannot do better than this
            return loc, "high", "original"  # best possible for this type
        else:
            return loc, "medium", "original"
    
    # Attempt 2: simplified (remove "de la", "del" etc.)
    simplified = simplify_address(address_clean)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            match_class = loc.raw.get("class")
            match_type = loc.raw.get("type")
            if match_class == "place" and match_type == "house":
                return loc, "high", "simplified"
            elif address_type == "street_only":
                return loc, "high", "simplified"
            else:
                return loc, "medium", "simplified"
    
    # Attempt 3 (for intersections): try first street alone
    if address_type == "intersection":
        street_a, street_b = split_intersection(address_clean)
        if street_a:
            loc = geocode_in_malaga(street_a)
            if loc:
                return loc, "low", "first_street_of_intersection"
            # Try simplified first street
            loc = geocode_in_malaga(simplify_address(street_a))
            if loc:
                return loc, "low", "first_street_simplified"
            # Try second street
            if street_b:
                loc = geocode_in_malaga(street_b)
                if loc:
                    return loc, "low", "second_street_of_intersection"
                loc = geocode_in_malaga(simplify_address(street_b))
                if loc:
                    return loc, "low", "second_street_simplified"
    
    # Attempt 4 (for house_number): strip house number, try as street
    if address_type == "house_number":
        no_number = strip_house_number(address_clean)
        if no_number != address_clean:
            loc = geocode_in_malaga(no_number)
            if loc:
                return loc, "low", "street_only_fallback"
            no_num_simplified = simplify_address(no_number)
            if no_num_simplified != no_number:
                loc = geocode_in_malaga(no_num_simplified)
                if loc:
                    return loc, "low", "street_only_simplified_fallback"
    
    return None, "failed", "none"


# Re-test
test_cases = [
    ("Avenida de la Paloma, 36, 29003, Málaga, Spain", "house_number"),
    ("Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain", "house_number"),
    ("Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Andarax and Camino de San Rafael, Málaga, Spain", "intersection"),
    ("Calle Agata and Calle Colmenarejo Nuevo, Málaga, Spain", "intersection"),
    ("Calle Almogía, Málaga, Spain", "street_only"),
]

print("Re-testing with intersection fallback:\n")
for addr, addr_type in test_cases:
    print(f"[{addr_type}]  {addr}")
    loc, quality, method = geocode_with_fallbacks(addr, addr_type)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  quality={quality}, method={method}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ FAILED")
    print()

Re-testing with intersection fallback:

[house_number]  Avenida de la Paloma, 36, 29003, Málaga, Spain
  ✓ (36.70166, -4.44376)  quality=high, method=simplified
    36, Avenida Paloma, San Carlos Condote, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, Espa

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70066, -4.44184)  quality=medium, method=simplified
    Avenida Paloma, Girón, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, España

[house_number]  Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain
  ✓ (36.70553, -4.43741)  quality=high, method=original
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

[house_number]  Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70023, -4.43988)  quality=medium, method=original
    Calle Federico García Lorca, 25 Años de Paz, Carretera de Cádi